In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
# Models
from models.unet import UNet
# Dataset, DataLoader and Transforms
from city_scapes.cityscapes_dataset import CityscapesDataset
from torch.utils.data import DataLoader
from pathlib import Path
import torchvision.transforms.functional as F
from PIL import Image
# For visualization
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

In [13]:
def compute_miou(pred, truth, num_classes, ignore_index=255):
    # TODO: Make computation run on GPU using torch 
    ious = []
    pred = pred.flatten()
    truth = truth.flatten()

    # Remove label values of ignore_index
    mask = truth != ignore_index
    pred = pred[mask]
    truth = truth[mask]

    for clas in range(num_classes):
        pred_clas = pred == clas
        truth_clas = truth == clas
        intersection = (pred_clas & truth_clas).sum().item()
        union = (pred_clas | truth_clas).sum().item()
        if union == 0:
            ious.append(float("nan"))
        else:
            ious.append(intersection/union)
    return np.nanmean(ious), ious

def compute_accuracy(pred, truth, ignore_index=255):
    # Remove label values of ignore_index
    mask = truth != ignore_index
    pred = pred[mask]
    truth = truth[mask]

    return (pred == truth).float().mean()

## Trainingsloop
In jedem Schritt:
- Eingabebilder und zugehörige Labels auf `device` laden
- Vorwärtsberechnung `forward pass` durchführen
- Verlust `loss` berechnen
- Gradienten mit `loss.backward()` berechnen
- Parameter mit `optimizer.step()` aktualisieren
- Metriken `loss`, `accuracy` und `MIoU` berechnen

In [14]:
def train(dataloader, model, loss_function, optimizer, device):
    # Switch to train mode
    model.train()

    # For storing running metrics
    epoch_loss, epoch_acc, epoch_miou = 0, 0, 0

    # Create a progress bar
    pbar = tqdm(dataloader, desc="Training loop", leave=False)
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        ### Forward pass
        # Get output/prediction
        output = model(images)
        # Calculate loss
        loss = loss_function(output, labels)

        ### Backpropagation
        # Gradients have to be zeroed for each batch
        optimizer.zero_grad()
        # Calculate the gradients
        loss.backward()
        # Update the parameters
        optimizer.step()

        ### Calculate metrics per batch
        prediction = output.argmax(dim=1)

        acc = compute_accuracy(prediction, labels)
        miou, _ = compute_miou(prediction, labels, model.num_classes)
        epoch_loss += loss.item()
        epoch_acc += acc.item()
        epoch_miou += miou

    # Return metrics averaged across all batches
    num_batches = len(dataloader)
    return epoch_loss/num_batches, epoch_acc/num_batches, epoch_miou/num_batches

## Validationloop
In jedem Schritt:
- Eingabebilder und zugehörige Labels auf `device` laden
- Vorwärtsberechnung `forward pass` durchführen
- Verlust `loss` berechnen
- Metriken `loss`, `accuracy` und `MIoU` berechnen

In [15]:
def valid(dataloader, model, loss_function, device):
    # Switch to eval mode
    model.eval()

    # For storing running metrics
    epoch_loss, epoch_acc, epoch_miou = 0, 0, 0

    # Create a progress bar
    pbar = tqdm(dataloader, desc="Validation loop", leave=False)
    # Disable gradient calculation
    with torch.no_grad():
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)

            ### Forward pass
            # Get output/prediction
            output = model(images)
            # Calculate loss
            loss = loss_function(output, labels)

            ### Calculate metrics per batch
            prediction = output.argmax(dim=1)

            acc = compute_accuracy(prediction, labels)
            miou, _ = compute_miou(prediction, labels, model.num_classes)
            epoch_loss += loss.item()
            epoch_acc += acc.item()
            epoch_miou += miou
    
    # Return metrics averaged across all batches
    num_batches = len(dataloader)
    return epoch_loss/num_batches, epoch_acc/num_batches, epoch_miou/num_batches
            
            

## Create the DataLoaders

In [17]:
### Transforms to resize images, labels
height, width = 256, 512
def image_transform(image):
    image = Image.fromarray(image)  # numpy -> PIL
    image = F.resize(image, (height, width), interpolation=F.InterpolationMode.BILINEAR) # Bilinear for RGB-Images
    image = F.pil_to_tensor(image).float() / 255.0  # Scaling
    return image


def label_transform(label):
    label = Image.fromarray(label)  # numpy -> PIL
    label = F.resize(label, (height, width), interpolation=F.InterpolationMode.NEAREST) # Nearest Neighbor for Labels
    label = F.pil_to_tensor(label).long().squeeze()  # PIL -> LongTensor no Scaling, squeeze() to remove channel dimension
    return label


# Define data paths
data_dir = Path("./data/cityscapes_intersections/fine_intersections_4")
image_dir = data_dir / "leftImg8bit_trainvaltest/leftImg8bit"
label_dir = data_dir / "gtFine_trainvaltest/gtFine"

# Initialize datasets
train_dataset = CityscapesDataset(image_dir, label_dir, split="train", image_transform=image_transform, label_transform=label_transform)
valid_dataset = CityscapesDataset(image_dir, label_dir, split="val", image_transform=image_transform, label_transform=label_transform)

# Create the dataloaders
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=16, shuffle=False)

for images, labels in train_dataloader:
    print(f"Images of first batch: {images.shape}")
    print(f"Labels of first batch: {labels.shape}")
    break

for images, labels in valid_dataloader:
    print(f"Images of first batch: {images.shape}")
    print(f"Labels of first batch: {labels.shape}")
    print(labels.dtype)
    print(labels.unique())
    break

Images of first batch: torch.Size([16, 3, 256, 512])
Labels of first batch: torch.Size([16, 256, 512])
Images of first batch: torch.Size([16, 3, 256, 512])
Labels of first batch: torch.Size([16, 256, 512])
torch.int64
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 11, 12, 13, 17, 18, 19, 20, 21,
        22, 23, 24, 25, 26, 27, 28, 31, 32, 33])


In [ ]:
### Hyperparameters
num_epochs = 10
learning_rate = 1e-4
# TODO: learning rate scheduler

save_at_epochs = 1

# Define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Initialize model and move to device
num_classes = 34    # TODO: Use trainIds
model = UNet(num_classes).to(device)
# Load checkpoint (optionally)
loaded_checkpoint = None
loaded_epochs = 0


# Define optimizer and loss function
loss_function = nn.CrossEntropyLoss(ignore_index=255)
optimizer = optim.Adam(params=model.parameters(), lr=learning_rate)

if loaded_checkpoint:
    model.load_state_dict(loaded_checkpoint["model_state_dict"])
    loaded_epochs = loaded_checkpoint["epoch"]
    optimizer.load_state_dict(loaded_checkpoint["optimizer_state_dict"])
    
best_val_loss = float(100_000)
pbar = tqdm(range(num_epochs))
for epoch in pbar:
    pbar.set_description(f"Epoch {epoch+1}/{num_epochs}")

    # Call train and validation loops
    train_loss, train_acc, train_miou = train(train_dataloader, model, loss_function, optimizer, device)
    val_loss, val_acc, val_miou = valid(valid_dataloader, model, loss_function, device)

    checkpoint = {
        "epoch": loaded_epochs + epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_acc": train_acc,
        "val_acc": val_acc,
        "train_miou": train_miou,
        "val_miou": val_miou
    }

    # Save the model every {save_at_epochs} epochs
    if epoch % save_at_epochs == 0:
        torch.save(checkpoint, f"./checkpoints/run/{model.name}_epoch_{loaded_epochs + epoch}")

    # Save the best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(checkpoint, f"./checkpoints/run/best_{model.name}")

    # Update the progress bar with metrics from current epoch
    pbar.set_postfix({'Train Loss':f'{train_loss:.3f}', 'Val Loss':f'{val_loss:.3f}', 'Train Acc':f'{train_acc:.3f}', 'Val Acc':f'{val_acc:.3f}', 
                      "Train MIoU":f"{train_miou:.3f}", "Val MIoU":f"{val_miou:.3f}"})
    # TODO: Logging to tensorboard